# Feature engineering

In [2]:
import pandas as pd
import numpy as np


In [17]:
train_df = pd.read_csv('../data/raw/train.csv')
test_df = pd.read_csv('../data/raw/test.csv')
train_df.head(5) 

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [24]:
def process(df, file_name):

    df_processed = df.copy()

    # dealing with NaNs
    df_processed['Age'] = df_processed['Age'].fillna(df['Age'].median())
    df_processed['Embarked'] = df_processed['Embarked'].fillna(df['Embarked'].mode()[0])
    df_processed['Fare'] = df_processed['Fare'].fillna(df['Fare'].median())


    # bins for Age and Fare
    df_processed["Age_bin"] = pd.cut(df_processed["Age"], 10, labels=False)
    df_processed["Fare_bin"] = pd.qcut(df_processed["Fare"], 13, labels=False, duplicates='drop')

    # TicketGroupSize feature
    df_processed['Ticket_group'] = df_processed.groupby('Ticket')['Ticket'].transform('count')

    # work with Initial
    df_processed['Initial'] = df_processed['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df_processed["Initial"] = df_processed["Initial"].replace({
                                            "Mlle": "Miss",
                                            "Mme": "Miss",
                                            "Ms": "Miss",
                                            "Dr": "Other",
                                            "Major": "Other",
                                            "Lady": "Other",
                                            "Countess": "Other",
                                            "Jonkheer": "Other",
                                            "Col": "Other",
                                            "Rev": "Other",
                                            "Capt": "Other",
                                            "Sir": "Mr",
                                            "Don": "Mr"
                                        })
    df_processed['Initial'] = df_processed['Initial'].map({
        'Mr': 0,
        'Mrs' : 1,
        'Miss' : 2,
        'Master' : 3,
        'Other' : 4
    })
    # family columns
    df_processed['Family_size'] = df_processed['SibSp'] + df_processed['Parch'] + 1
    df_processed['Is_alone'] = (df_processed['Family_size'] == 1).astype(int)
    
    # Deck feature
    df_processed["Deck"] = df_processed["Cabin"].str[0].fillna("Unknown")

    df_processed = pd.get_dummies(
    df_processed,
    columns=["Embarked", "Sex", "Deck"],
    drop_first=True
    )

    df_processed = df_processed.drop(
        columns=['Name', 'Ticket', 'Cabin', 'PassengerId', 'SibSp', 'Parch']
    )

    return df_processed.to_csv(f'../data/processed/{file_name}.csv')

In [25]:
process(train_df, file_name='train')
process(test_df, file_name='test')


In [26]:
train_processed = pd.read_csv("../data/processed/train.csv")
test_processed = pd.read_csv("../data/processed/test.csv")

target = train_processed["Survived"]
train_features = train_processed.drop("Survived", axis=1)

train_features, test_processed = train_features.align(
    test_processed,
    join="left",
    axis=1,
    fill_value=0
)

train_processed = pd.concat([target, train_features], axis=1)

train_processed.to_csv("../data/processed/train.csv", index=False)
test_processed.to_csv("../data/processed/test.csv", index=False)